In [1]:
%matplotlib inline

import os
import sys

sys.path.append('../../../../')

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [2]:
from __future__ import annotations
from typing import Optional, Union, Sequence, Callable, Mapping

import torch
from torch.nn.modules.utils import _pair

import numpy as np

%load_ext autoreload
%autoreload 2

from computer_vision.slowfast.mmaction.datasets.transforms.loading import DecordInit, SampleFrames, DecordDecode
from computer_vision.slowfast.mmaction.datasets.transforms.processing import Resize, _init_lazy_of_proper, RandomCrop, CenterCrop, ThreeCrop, \
RandomResizedCrop, Flip
from computer_vision.slowfast.mmaction.datasets.transforms.formatting import FormatShape, PackActionInputs
from computer_vision.slowfast.mmengine.utils.misc import is_tuple_of
from computer_vision.slowfast.mmcv.image.geometric import imflip
from computer_vision.slowfast.mmcv.image.photometric import iminvert
from computer_vision.slowfast.mmaction.evaluation.metrics.acc_metric import to_tensor
from computer_vision.slowfast.mmaction.structures.action_data_sample import ActionDataSample
from computer_vision.slowfast.mmengine.structures.instance_data import InstanceData
from computer_vision.slowfast.mmengine.dataset.base_dataset import Compose
from computer_vision.slowfast.mmengine.dataset.utils import pseudo_collate

In [3]:
output_dirpath='D:/results/ucf101'
video_fpath=f'{output_dirpath}/demo.mp4'
assert os.path.isfile(video_fpath)

data=dict(filename=video_fpath, label=-1, start_index=0, modality='RGB')
print(f"{data=}")

transforms=[DecordInit(io_backend='disk'),
            SampleFrames(clip_len=32, frame_interval=2, num_clips=10, test_mode=True),
            DecordDecode(),
            Resize(scale=(-1, 256), keep_ratio=True, interpolation='bilinear', lazy=False),
            RandomResizedCrop(),
            Resize(scale=(224, 224), keep_ratio=False, interpolation='bilinear', lazy=False),
            Flip(flip_ratio=1.),
            FormatShape(input_format='NCTHW'),
            PackActionInputs()]
transforms=Compose(transforms)
results=transforms(data)
print(f"{results.keys()=}")
print(f"{type(results['inputs'])=}, {results['inputs'].dtype=}, {results['inputs'].shape=}")
print("results['data_samples']=", results['data_samples'])
print(f"{results['data_samples'].get('gt_label')=}, {results['data_samples'].get('img_shape')=}")

data={'filename': 'D:/results/ucf101/demo.mp4', 'label': -1, 'start_index': 0, 'modality': 'RGB'}
results.keys()=dict_keys(['inputs', 'data_samples'])
type(results['inputs'])=<class 'torch.Tensor'>, results['inputs'].dtype=torch.uint8, results['inputs'].shape=torch.Size([10, 3, 32, 224, 224])
results['data_samples']= <ActionDataSample(

    META INFORMATION
    img_shape:(224, 224)

    DATA FIELDS
    gt_label:tensor([-1])
) at 0x1c610997b20>
results['data_samples'].get('gt_label')=tensor([-1]), results['data_samples'].get('img_shape')=(224, 224)


In [13]:
data = pseudo_collate([results])
print(f"{type(data['inputs'])=}, {len(data['inputs'])=}, {[x.shape for x in data['inputs']]=}")
print(f"{data['data_samples']=}")

type(data['inputs'])=<class 'list'>, len(data['inputs'])=1, [x.shape for x in data['inputs']]=[torch.Size([10, 3, 32, 224, 224])]
data['data_samples']=[<ActionDataSample(

    META INFORMATION
    img_shape:(224, 224)

    DATA FIELDS
    gt_label:tensor([-1])
) at 0x1c610997b20>]
